In [1]:
import pandas as pd
import numpy as np

In [27]:
import matplotlib.pyplot as plt

In [3]:
data = pd.read_csv(r'data\NF_ToN_IoT.csv')

In [4]:
data.columns


Index(['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES',
       'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS',
       'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN',
       'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT',
       'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
       'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
       'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS',
       'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS',
       'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
       'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
       'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
       'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
       'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
       'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label', 'Attack'],
      dtype='str')

In [5]:
data = data.drop_duplicates()
data = data.dropna()
data = data.reset_index(drop=True)

In [6]:
data["Attack"].value_counts()

Attack
Benign        3601284
scanning      3002169
xss           2449955
ddos          1746590
password       993718
injection      660467
dos            654359
backdoor        16259
mitm             7723
ransomware       3357
Name: count, dtype: int64

In [15]:
data.head()

,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
0,49235,4444,6,0.0,155392,202,34552,149,24,24,...,45555,4805,0,0,0,0,0,0,1,ransomware
1,49228,1880,6,0.0,1600,40,35741,65,24,16,...,16425,237,0,0,0,0,0,0,0,Benign
2,0,0,1,0.0,212,2,0,0,0,0,...,0,0,771,3,0,0,0,0,0,Benign
3,65317,1900,17,0.0,165,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Benign
4,60766,15600,17,0.0,63,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Benign


In [67]:
data.dtypes

L4_SRC_PORT                      int64
L4_DST_PORT                      int64
PROTOCOL                         int64
L7_PROTO                       float64
IN_BYTES                         int64
IN_PKTS                          int64
OUT_BYTES                        int64
OUT_PKTS                         int64
TCP_FLAGS                        int64
CLIENT_TCP_FLAGS                 int64
SERVER_TCP_FLAGS                 int64
FLOW_DURATION_MILLISECONDS       int64
DURATION_IN                      int64
DURATION_OUT                     int64
MIN_TTL                          int64
MAX_TTL                          int64
LONGEST_FLOW_PKT                 int64
SHORTEST_FLOW_PKT                int64
MIN_IP_PKT_LEN                   int64
MAX_IP_PKT_LEN                   int64
SRC_TO_DST_SECOND_BYTES        float64
DST_TO_SRC_SECOND_BYTES        float64
RETRANSMITTED_IN_BYTES           int64
RETRANSMITTED_IN_PKTS            int64
RETRANSMITTED_OUT_BYTES          int64
RETRANSMITTED_OUT_PKTS   

In [20]:
Zero_day = "ransomware"

train_data = data[data["Attack"] != Zero_day]

In [21]:
train_data["Attack"].value_counts()

Attack
Benign       3601284
scanning     3002169
xss          2449955
ddos         1746590
password      993718
injection     660467
dos           654359
backdoor       16259
mitm            7723
Name: count, dtype: int64

In [33]:
train_df = train_data.sample(frac=0.6, random_state=42)

In [34]:
train_df["Attack"].value_counts()

Attack
Benign       2159201
scanning     1802068
xss          1469396
ddos         1048325
password      596370
injection     396000
dos           393715
backdoor        9766
mitm            4673
Name: count, dtype: int64

In [35]:
remaining_data = data.drop(train_df.index)

test_df = remaining_data[(remaining_data["Attack"] == Zero_day) | (remaining_data["Attack"] == "Benign")]

In [36]:
test_df["Attack"].value_counts()

Attack
Benign        1442083
ransomware       3357
Name: count, dtype: int64

In [37]:
train_df.groupby("Attack")["Label"].unique()

Attack
Benign       [0]
backdoor     [1]
ddos         [1]
dos          [1]
injection    [1]
mitm         [1]
password     [1]
scanning     [1]
xss          [1]
Name: Label, dtype: object

In [38]:
train_df["Label"].value_counts()

Label
1    5720313
0    2159201
Name: count, dtype: int64

In [39]:
x_train = train_df.drop(columns=["Attack", "Label"])
y_train = train_df["Label"]
x_test = test_df.drop(columns=["Attack", "Label"])
y_test = test_df["Label"]

In [46]:
x_train = np.log1p(x_train)
x_test = np.log1p(x_test)

In [45]:
np.isinf(x_train).sum().value_counts()

0    41
Name: count, dtype: int64

In [47]:
x_train.max()

L4_SRC_PORT                     11.090355
L4_DST_PORT                     11.090355
PROTOCOL                         4.077537
L7_PROTO                         5.517453
IN_BYTES                        19.487989
IN_PKTS                         13.058959
OUT_BYTES                       18.918231
OUT_PKTS                        12.926115
TCP_FLAGS                        5.411646
CLIENT_TCP_FLAGS                 5.411646
SERVER_TCP_FLAGS                 5.411646
FLOW_DURATION_MILLISECONDS      15.272954
DURATION_IN                     11.691699
DURATION_OUT                    11.691565
MIN_TTL                          5.545177
MAX_TTL                          5.545177
LONGEST_FLOW_PKT                11.090355
SHORTEST_FLOW_PKT                8.291797
MIN_IP_PKT_LEN                   6.393591
MAX_IP_PKT_LEN                  11.090355
SRC_TO_DST_SECOND_BYTES          6.400405
DST_TO_SRC_SECOND_BYTES        592.391464
RETRANSMITTED_IN_BYTES           2.792612
RETRANSMITTED_IN_PKTS            2

In [48]:
from sklearn.preprocessing import MinMaxScaler

In [49]:
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [50]:
x_train_scaled.max(), x_train_scaled.min()

(np.float64(1.0000000000000002), np.float64(0.0))

In [51]:
x_test_scaled.max(), x_test_scaled.min()

(np.float64(1.0242481864775959), np.float64(0.0))

In [52]:
window_size = 10

In [55]:
x_train_scaled = x_train_scaled.astype("float32")
x_test_scaled  = x_test_scaled.astype("float32")

In [53]:
def create_sequences(X, y, window):

    X_seq = []
    y_seq = []

    for i in range(len(X) - window):
        X_seq.append(X[i:i+window])
        y_seq.append(y[i+window-1])

    return np.array(X_seq), np.array(y_seq)



In [54]:
X_train_seq, y_train_seq = create_sequences(
    x_train_scaled,
    y_train.values,
    window_size
)

MemoryError: Unable to allocate 24.1 GiB for an array with shape (7879504, 10, 41) and data type float64

In [58]:
from tensorflow.keras.utils import Sequence

In [59]:
class DataGenerator(Sequence):
    def __init__(self,x,y,window_size,batch_size):
        self.x = x
        self.y = y
        self.window_size = window_size
        self.batch_size = batch_size
    
    def __len__(self):
        return (len(self.x) - self.window_size) // self.batch_size
    
    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = start + self.batch_size
        
        x_batch = []
        y_batch = []
        for i in range(start, end):
            x_batch.append(self.x[i:i+self.window_size])
            y_batch.append(self.y[i+self.window_size-1])
            
        return np.array(x_batch), np.array(y_batch)

In [60]:
window_size = 10
batch_size = 256

train_gen = DataGenerator(x_train_scaled, y_train.values, window_size, batch_size)
test_gen = DataGenerator(x_test_scaled, y_test.values, window_size, batch_size)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout

In [62]:
from tensorflow.keras.layers import MaxPooling1D

In [64]:
model = Sequential()

model.add(
    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        input_shape=(window_size, x_train_scaled.shape[1])
    )
)

model.add(MaxPooling1D(pool_size=2))

model.add(LSTM(64))

model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [65]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [81]:
from sklearn.utils.class_weight import compute_class_weight

In [82]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print(class_weights)

{np.int64(0): np.float64(1.8246365206388844), np.int64(1): np.float64(0.6887310187397089)}


In [83]:
from tensorflow.keras.callbacks import EarlyStopping

In [84]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

In [85]:
model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
30779/30779 ━━━━━━━━━━━━━━━━━━━━ 215s 7ms/step - accuracy: 0.9790 - loss: 0.0596 - val_accuracy: 0.9682 - val_loss: 0.0735
Epoch 2/10
30779/30779 ━━━━━━━━━━━━━━━━━━━━ 222s 7ms/step - accuracy: 0.9792 - loss: 0.0590 - val_accuracy: 0.9773 - val_loss: 0.0618
Epoch 3/10
30779/30779 ━━━━━━━━━━━━━━━━━━━━ 226s 7ms/step - accuracy: 0.9793 - loss: 0.0585 - val_accuracy: 0.9702 - val_loss: 0.0677
Epoch 4/10
30779/30779 ━━━━━━━━━━━━━━━━━━━━ 205s 7ms/step - accuracy: 0.9795 - loss: 0.0581 - val_accuracy: 0.9713 - val_loss: 0.0659
Epoch 5/10
30779/30779 ━━━━━━━━━━━━━━━━━━━━ 207s 7ms/step - accuracy: 0.9796 - loss: 0.0578 - val_accuracy: 0.9682 - val_loss: 0.0764
Epoch 6/10
30779/30779 ━━━━━━━━━━━━━━━━━━━━ 204s 7ms/step - accuracy: 0.9798 - loss: 0.0574 - val_accuracy: 0.9719 - val_loss: 0.0657
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 2.


In [111]:
y_prob = model.predict(test_gen)

5646/5646 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step


In [87]:
from sklearn.metrics import classification_report, confusion_matrix

In [105]:
y_true = []
for _, y_batch in test_gen:
    y_true.extend(y_batch)
y_true = np.array(y_true)

In [112]:
y_true = y_true[:len(y_prob)]

In [108]:
len(y_true), len(y_pred_bin)

(1445376, 1445376)

In [113]:
for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
    print(f"\nThreshold = {t}")
    y_pred = (y_prob > t).astype(int)
    print(classification_report(y_true, y_pred,
                                target_names=["Benign","Attack"]))


Threshold = 0.1
              precision    recall  f1-score   support

      Benign       1.00      0.93      0.96   1442053
      Attack       0.01      0.32      0.02      3323

    accuracy                           0.93   1445376
   macro avg       0.50      0.62      0.49   1445376
weighted avg       1.00      0.93      0.96   1445376


Threshold = 0.2
              precision    recall  f1-score   support

      Benign       1.00      0.95      0.98   1442053
      Attack       0.01      0.18      0.02      3323

    accuracy                           0.95   1445376
   macro avg       0.50      0.57      0.50   1445376
weighted avg       1.00      0.95      0.97   1445376


Threshold = 0.3
              precision    recall  f1-score   support

      Benign       1.00      0.97      0.98   1442053
      Attack       0.01      0.10      0.01      3323

    accuracy                           0.96   1445376
   macro avg       0.50      0.53      0.50   1445376
weighted avg       1.00

In [80]:
y_train.value_counts()

Label
1    5720313
0    2159201
Name: count, dtype: int64